In [1]:
from pathlib import Path
import pandas as pd
import duckdb

DATA_DIR = Path("/datasets/_deepnote_work/mahmad")

con = duckdb.connect()

print("تم تجهيز البيئة")

تم تجهيز البيئة


In [2]:
# الملفات

files = sorted(DATA_DIR.glob("*.csv"))

file_catalog = pd.DataFrame({
    "file_name": [f.name for f in files],
    "size_mb": [
        round(f.stat().st_size / (1024 ** 2), 2)
        for f in files
    ],
    "full_path": [str(f) for f in files]
})

print(f"عدد الملفات: {len(files)}")

display(file_catalog)

عدد الملفات: 7


,file_name,size_mb,full_path
0,customers.csv,0.33,/datasets/_deepnote_work/mahmad/customers.csv
1,daily_growth.csv,0.00,/datasets/_deepnote_work/mahmad/daily_growth.csv
2,daily_revenue.csv,0.00,/datasets/_deepnote_work/mahmad/daily_revenue.csv
3,driver_insights.csv,0.03,/datasets/_deepnote_work/mahmad/driver_insight...
4,drivers.csv,0.05,/datasets/_deepnote_work/mahmad/drivers.csv
5,final_clean_trips.csv,779.79,/datasets/_deepnote_work/mahmad/final_clean_tr...
6,locations.csv,0.00,/datasets/_deepnote_work/mahmad/locations.csv


In [3]:
# المصادر

source_files = {
    file.stem.lower(): str(file)
    for file in files
}

main_file = source_files["final_clean_trips"]

customers = pd.read_csv(
    source_files["customers"]
)

drivers = pd.read_csv(
    source_files["drivers"]
)

locations = pd.read_csv(
    source_files["locations"]
)

daily_revenue = pd.read_csv(
    source_files["daily_revenue"]
)

daily_growth = pd.read_csv(
    source_files["daily_growth"]
)

driver_insights = pd.read_csv(
    source_files["driver_insights"]
)

source_roles = pd.DataFrame({
    "source": [
        "final_clean_trips",
        "customers",
        "drivers",
        "locations",
        "daily_revenue",
        "daily_growth",
        "driver_insights"
    ],
    "role": [
        "الرحلات",
        "العملاء",
        "السائقون",
        "المدن",
        "الإيراد اليومي",
        "النمو اليومي",
        "مؤشرات السائقين"
    ]
})

display(source_roles)

con.execute(
    f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT *
    FROM read_csv_auto(
        '{main_file}',
        HEADER = TRUE
    )
    """
)

print("تم تجهيز المصادر")

,source,role
0,final_clean_trips,الرحلات
1,customers,العملاء
2,drivers,السائقون
3,locations,المدن
4,daily_revenue,الإيراد اليومي
5,daily_growth,النمو اليومي
6,driver_insights,مؤشرات السائقين


تم تجهيز المصادر


In [4]:
# Baseline

baseline = con.execute("""
    SELECT
        COUNT(*) AS total_trips,

        COUNT(*) FILTER (
            WHERE status = 'COMPLETED'
        ) AS completed_trips,

        COUNT(*) FILTER (
            WHERE status = 'CANCELLED'
        ) AS cancelled_trips,

        COUNT(*) FILTER (
            WHERE status = 'FAILED'
        ) AS failed_trips,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE status = 'COMPLETED'
            ) / COUNT(*),
            2
        ) AS completion_rate,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE status = 'CANCELLED'
            ) / COUNT(*),
            2
        ) AS cancellation_rate,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE status = 'FAILED'
            ) / COUNT(*),
            2
        ) AS failure_rate,

        ROUND(SUM(fare_amount), 2) AS total_fare,

        ROUND(AVG(fare_amount), 2) AS average_fare,

        ROUND(AVG(distance_km), 2) AS average_distance_km,

        ROUND(AVG(price_per_km), 2) AS average_price_per_km,

        COUNT(DISTINCT customer_id) AS active_customers,

        COUNT(DISTINCT driver_id) AS active_drivers,

        COUNT(DISTINCT city) AS active_cities,

        MIN(start_time) AS first_trip,

        MAX(start_time) AS last_trip

    FROM trips
""").df()

print("Baseline")

display(baseline)

Baseline


,total_trips,completed_trips,cancelled_trips,failed_trips,completion_rate,cancellation_rate,failure_rate,total_fare,average_fare,average_distance_km,average_price_per_km,active_customers,active_drivers,active_cities,first_trip,last_trip
0,5960965,3574865,1193226,1192874,59.97,20.02,20.01,2.847391e+10,4776.73,250.23,67.95,5000,1000,4,2026-01-01,2026-01-12 13:46:39


In [5]:
# EDA

eda = con.execute("""
    SELECT
        status,
        COUNT(*) AS trips,

        ROUND(
            100.0 * COUNT(*) /
            SUM(COUNT(*)) OVER (),
            2
        ) AS trip_share,

        ROUND(AVG(fare_amount), 2) AS average_fare,

        ROUND(AVG(distance_km), 2) AS average_distance,

        ROUND(AVG(price_per_km), 2) AS average_price_per_km

    FROM trips

    GROUP BY status

    ORDER BY trips DESC
""").df()

print("EDA")

display(eda)

EDA


,status,trips,trip_share,average_fare,average_distance,average_price_per_km
0,COMPLETED,3574865,59.97,4795.21,250.26,67.71
1,CANCELLED,1193226,20.02,4715.31,250.18,68.06
2,FAILED,1192874,20.01,4782.76,250.21,68.56


In [6]:
# Time & Growth

daily_revenue["trip_date"] = pd.to_datetime(
    daily_revenue["trip_date"]
)

daily_growth["trip_date"] = pd.to_datetime(
    daily_growth["trip_date"]
)

revenue_by_day = (
    daily_revenue
    .groupby("trip_date", as_index=False)
    .agg(
        total_trips=("daily_trips", "sum"),
        total_revenue=("daily_revenue", "sum")
    )
    .sort_values("trip_date")
)

growth_summary = (
    daily_growth
    .groupby("trip_date", as_index=False)
    .agg(
        average_growth=("growth_percentage", "mean")
    )
    .sort_values("trip_date")
)

time_analysis = revenue_by_day.merge(
    growth_summary,
    on="trip_date",
    how="left"
)

print("Time & Growth")

display(time_analysis)

Time & Growth


,trip_date,total_trips,total_revenue,average_growth
0,2026-01-01,514350,2.458758e+09,NaN
1,2026-01-02,515090,2.461936e+09,0.0900
2,2026-01-03,514230,2.443588e+09,-0.8575
3,2026-01-04,515456,2.465932e+09,1.6050
4,2026-01-05,515376,2.515239e+09,2.1500
5,2026-01-06,514679,2.500221e+09,-0.6825
6,2026-01-07,515550,2.433457e+09,-3.1100
7,2026-01-08,515829,2.453744e+09,1.2875
8,2026-01-09,514733,2.478788e+09,1.0325
9,2026-01-10,514674,2.455838e+09,-1.1925


In [7]:
# العملاء

con.register(
    "customers_data",
    customers
)

customer_analysis = con.execute("""
    SELECT
        c.customer_id,

        COUNT(t.trip_id) AS total_trips,

        COUNT(t.trip_id) FILTER (
            WHERE t.status = 'COMPLETED'
        ) AS completed_trips,

        COUNT(t.trip_id) FILTER (
            WHERE t.status = 'CANCELLED'
        ) AS cancelled_trips,

        COUNT(t.trip_id) FILTER (
            WHERE t.status = 'FAILED'
        ) AS failed_trips,

        ROUND(
            100.0 * COUNT(t.trip_id) FILTER (
                WHERE t.status = 'COMPLETED'
            ) / NULLIF(COUNT(t.trip_id), 0),
            2
        ) AS completion_rate,

        ROUND(
            100.0 * COUNT(t.trip_id) FILTER (
                WHERE t.status = 'CANCELLED'
            ) / NULLIF(COUNT(t.trip_id), 0),
            2
        ) AS cancellation_rate,

        ROUND(
            SUM(t.fare_amount),
            2
        ) AS total_fare,

        ROUND(
            AVG(t.fare_amount),
            2
        ) AS average_fare,

        MIN(t.start_time) AS first_trip_time,

        MAX(t.start_time) AS last_trip_time

    FROM customers_data c

    LEFT JOIN trips t
        ON c.customer_id = t.customer_id

    GROUP BY
        c.customer_id

    ORDER BY
        total_fare DESC
""").df()

print("Customers")

display(
    customer_analysis.head(20)
)

Customers


,customer_id,total_trips,completed_trips,cancelled_trips,failed_trips,completion_rate,cancellation_rate,total_fare,average_fare,first_trip_time,last_trip_time
0,CST_1343,1224,752,229,243,61.44,18.71,14842903.53,12126.56,2026-01-01 00:00:19,2026-01-12 13:43:57
1,CST_329,1175,705,223,247,60.00,18.98,13772977.51,11721.68,2026-01-01 00:02:34,2026-01-12 13:39:38
2,CST_358,1251,725,265,261,57.95,21.18,13584409.37,10858.84,2026-01-01 00:32:52,2026-01-12 13:41:32
3,CST_3906,1158,682,242,234,58.89,20.90,13479717.51,11640.52,2026-01-01 00:08:42,2026-01-12 13:25:41
4,CST_3876,1235,722,243,270,58.46,19.68,12751956.95,10325.47,2026-01-01 00:03:14,2026-01-12 13:26:11
5,CST_1155,1195,727,239,229,60.84,20.00,12708835.81,10635.01,2026-01-01 00:45:11,2026-01-12 13:05:40
6,CST_2265,1208,740,221,247,61.26,18.29,12699771.70,10513.06,2026-01-01 00:04:53,2026-01-12 13:37:56
7,CST_4508,1192,684,252,256,57.38,21.14,12612051.56,10580.58,2026-01-01 00:00:27,2026-01-12 13:46:36
8,CST_2653,1257,729,227,301,58.00,18.06,12568352.14,9998.69,2026-01-01 00:26:17,2026-01-12 13:31:28
9,CST_2964,1245,764,252,229,61.37,20.24,12545702.74,10076.87,2026-01-01 00:01:54,2026-01-12 13:31:56


In [8]:
# السائقون

con.register(
    "drivers_data",
    drivers
)

driver_analysis = con.execute("""
    SELECT
        d.driver_id,

        COUNT(t.trip_id) AS total_trips,

        COUNT(t.trip_id) FILTER (
            WHERE t.status = 'COMPLETED'
        ) AS completed_trips,

        COUNT(t.trip_id) FILTER (
            WHERE t.status = 'CANCELLED'
        ) AS cancelled_trips,

        COUNT(t.trip_id) FILTER (
            WHERE t.status = 'FAILED'
        ) AS failed_trips,

        ROUND(
            100.0 * COUNT(t.trip_id) FILTER (
                WHERE t.status = 'COMPLETED'
            ) / NULLIF(COUNT(t.trip_id), 0),
            2
        ) AS completion_rate,

        ROUND(
            100.0 * COUNT(t.trip_id) FILTER (
                WHERE t.status = 'CANCELLED'
            ) / NULLIF(COUNT(t.trip_id), 0),
            2
        ) AS cancellation_rate,

        ROUND(
            100.0 * COUNT(t.trip_id) FILTER (
                WHERE t.status = 'FAILED'
            ) / NULLIF(COUNT(t.trip_id), 0),
            2
        ) AS failure_rate,

        ROUND(
            SUM(t.fare_amount),
            2
        ) AS total_fare,

        ROUND(
            AVG(t.fare_amount),
            2
        ) AS average_fare,

        ROUND(
            AVG(t.distance_km),
            2
        ) AS average_distance_km

    FROM drivers_data d

    LEFT JOIN trips t
        ON d.driver_id = t.driver_id

    GROUP BY
        d.driver_id

    ORDER BY
        completed_trips DESC
""").df()


driver_analysis = (
    driver_analysis
    .merge(
        driver_insights,
        on="driver_id",
        how="left",
        suffixes=("", "_insight")
    )
)

print("Drivers")

display(
    driver_analysis.head(20)
)


Drivers


,driver_id,total_trips,completed_trips,cancelled_trips,failed_trips,completion_rate,cancellation_rate,failure_rate,total_fare,average_fare,average_distance_km,completed_trips_insight,total_earnings,earnings_rank,activity_quartile
0,DRV_614,6174,3765,1191,1218,60.98,19.29,19.73,28769162.76,4659.73,248.99,3765,19697015.28,233,1
1,DRV_51,6140,3750,1185,1205,61.07,19.30,19.63,23804730.48,3876.99,249.63,3750,15375492.36,691,1
2,DRV_472,6247,3748,1249,1250,60.00,19.99,20.01,24525684.29,3925.99,252.27,3748,13912461.24,832,1
3,DRV_955,6161,3738,1271,1152,60.67,20.63,18.70,23804179.21,3863.69,250.90,3738,14767698.20,751,1
4,DRV_506,6116,3733,1146,1237,61.04,18.74,20.23,28009491.82,4579.71,250.35,3733,16454735.52,557,1
5,DRV_327,6057,3729,1143,1185,61.57,18.87,19.56,34747230.14,5736.71,245.70,3729,18362522.97,346,1
6,DRV_82,6116,3722,1191,1203,60.86,19.47,19.67,20504542.85,3352.61,251.68,3722,11439707.59,971,1
7,DRV_848,6090,3718,1202,1170,61.05,19.74,19.21,29551405.82,4852.45,250.96,3718,13095709.79,901,1
8,DRV_329,6114,3717,1215,1182,60.79,19.87,19.33,30734606.95,5026.92,248.68,3717,19863618.44,223,1
9,DRV_691,6103,3711,1220,1172,60.81,19.99,19.20,29375171.24,4813.23,251.64,3711,20229267.15,191,1


In [9]:
# المدن

city_analysis = (
    daily_revenue
    .groupby("city", as_index=False)
    .agg(
        total_trips=("daily_trips", "sum"),
        total_revenue=("daily_revenue", "sum"),
        average_daily_trips=("daily_trips", "mean"),
        average_daily_revenue=("daily_revenue", "mean")
    )
)

city_analysis["revenue_share"] = (
    100
    * city_analysis["total_revenue"]
    / city_analysis["total_revenue"].sum()
).round(2)

city_analysis = (
    city_analysis
    .merge(
        locations,
        on="city",
        how="left",
        suffixes=("", "_location")
    )
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

print("Cities")

display(city_analysis)

Cities


,city,total_trips,total_revenue,average_daily_trips,average_daily_revenue,revenue_share,total_trips_location,completed_trips,cancelled_trips,failed_trips,total_revenue_location,average_fare,average_distance
1,Khartoum,2385855,1.132278e+10,198821.250000,9.435651e+08,39.77,2385855,1429468,478580,477807,1.132278e+10,4745.80,250.22
3,Port Sudan,1191711,5.731530e+09,99309.250000,4.776275e+08,20.13,1191711,716047,238026,237638,5.731530e+09,4809.50,250.25
0,Bahri,1192566,5.727200e+09,99380.500000,4.772667e+08,20.11,1192566,715459,238461,238646,5.727200e+09,4802.42,250.28
2,Omdurman,1190833,5.692399e+09,99236.083333,4.743666e+08,19.99,1190833,713891,238159,238783,5.692399e+09,4780.18,250.19


In [10]:
# Insights

correlation_sample = con.execute("""
    SELECT
        fare_amount,
        distance_km,
        price_per_km
    FROM trips
    USING SAMPLE 200000
""").df()

correlation = (
    correlation_sample
    .corr()
    .round(2)
)

top_city = city_analysis.iloc[0]

top_customer = customer_analysis.iloc[0]

top_driver = driver_analysis.iloc[0]

insights = pd.DataFrame({
    "insight": [
        "أكبر مدينة بالإيراد",
        "أعلى عميل بالإيراد",
        "أعلى سائق بالرحلات المكتملة",
        "ارتباط الأجرة بالمسافة",
        "ارتباط الأجرة بالسعر لكل كيلومتر"
    ],
    "value": [
        top_city["city"],
        top_customer["customer_id"],
        top_driver["driver_id"],
        correlation.loc["fare_amount", "distance_km"],
        correlation.loc["fare_amount", "price_per_km"]
    ]
})

print("Insights")

display(insights)

Insights


,insight,value
0,أكبر مدينة بالإيراد,Khartoum
1,أعلى عميل بالإيراد,CST_1343
2,أعلى سائق بالرحلات المكتملة,DRV_614
3,ارتباط الأجرة بالمسافة,0.0
4,ارتباط الأجرة بالسعر لكل كيلومتر,0.26


In [21]:
# BI Outputs

bi_folder = Path("/work/bi_outputs")
bi_folder.mkdir(parents=True, exist_ok=True)

bi_tables = {
    "customer_analysis": customer_analysis,
    "driver_analysis": driver_analysis,
    "city_analysis": city_analysis,
    "daily_revenue": daily_revenue,
    "daily_growth": daily_growth,
    "driver_insights": driver_insights
}

for name, df in bi_tables.items():
    df.to_csv(
        bi_folder / f"{name}.csv",
        index=False,
        encoding="utf-8-sig"
    )

print(f"تم تجهيز {len(bi_tables)} ملفات")

تم تجهيز 6 ملفات


In [22]:
# Reconciliation

checks = pd.DataFrame({
    "metric": [
        "total_trips",
        "completed_trips",
        "cancelled_trips",
        "failed_trips",
        "customers",
        "drivers",
        "cities"
    ],
    "value": [
        baseline.loc[0, "total_trips"],
        baseline.loc[0, "completed_trips"],
        baseline.loc[0, "cancelled_trips"],
        baseline.loc[0, "failed_trips"],
        customer_analysis["customer_id"].nunique(),
        driver_analysis["driver_id"].nunique(),
        city_analysis["city"].nunique()
    ]
})

display(checks)

print("تمت المطابقة")

,metric,value
0,total_trips,5960965
1,completed_trips,3574865
2,cancelled_trips,1193226
3,failed_trips,1192874
4,customers,5000
5,drivers,1000
6,cities,4


تمت المطابقة


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=cbe6893b-8eab-4f7b-91b2-c92f568a73aa' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>